Authentification

In [ ]:
# token
import os

os.environ["DAGSHUB_TOKEN"] = "xxxxx"

# Install MLFlow & the DagsHub python client

In [ ]:
%pip install -q dagshub mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.0/623.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

# Use the DagsHub client to setup connection information for MLflow

In [ ]:
pip install python-dotenv

In [ ]:
import dagshub
dagshub.auth.add_app_token(os.getenv("DAGSHUB_TOKEN"))

In [ ]:
import os
import mlflow
import pandas as pd
import time
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Initialisation Dagshub
dagshub.init(repo_owner='ungjeanclaude', repo_name='early-prediction-alzheimer', mlflow=True)

# Fin de la session MLflow précédente
mlflow.end_run()

from mlflow.tracking import MlflowClient

client = MlflowClient()

#experiment_name = "base_model_99"
experiment = mlflow.set_experiment("base_model_13")

print(experiment)

with mlflow.start_run():
    print("Début de l'entraînement du modèle...")

    start_time = time.time()

    # Activation du suivi automatique
    mlflow.sklearn.autolog(log_models=True)

    from google.colab import drive
    drive.mount('/content/drive')
    # Data manipulation
    import librosa
    import numpy as np
    import pandas as pd
    import os

    # Machine Learning & Deep Learning
    import tensorflow as tf
    from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.utils import to_categorical
    from tensorflow.image import resize
    from tensorflow.keras.models import load_model

    # Data Visualization
    import librosa.display
    import matplotlib.pyplot as plt

    # Image Processing
    from skimage.transform import resize

    labels_file = "/content/drive/MyDrive/final_project_alz/DATA/train_labels.csv"  # Path to CSV file
    audio_dir = "/content/drive/MyDrive/final_project_alz/DATA/wav_files_train"     # Path to audio files

    # Load labels
    labels_df = pd.read_csv(labels_file)

    # Get classes (diagnosis_control, diagnosis_mci, diagnosis_adrd)
    classes = ['diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd']

    def load_and_preprocess_data(audio_dir, labels_df, target_shape=(128, 128)):
        """
        Charge et prétraite les fichiers audio.
        Convertit chaque fichier en spectrogramme Mel.
        """
        data = []
        labels = []

        for idx, row in labels_df.iterrows():
            # Identify the audio file and its labels
            uid = row['uid']
            file_path = os.path.join(audio_dir, f"{uid}.wav")

            # Check if audio file exists
            if not os.path.exists(file_path):
                print(f"Fichier audio manquant : {file_path}")
                continue

            # Load audio
            audio_data, sample_rate = librosa.load(file_path, sr=None)

            # Convert to Mel spectrogram
            mel_spectrogram = librosa.feature.melspectrogram(y=audio_data, sr=sample_rate)
            mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)  # Décibels
            mel_spectrogram = np.expand_dims(mel_spectrogram, axis=-1)          # Add a dimension for Conv2D

            # Resize to a fixed size (e.g., 128x128)
            mel_spectrogram = resize(mel_spectrogram, target_shape)

            # Add data and labels
            data.append(mel_spectrogram)
            labels.append([row['diagnosis_control'], row['diagnosis_mci'], row['diagnosis_adrd']])

        return np.array(data), np.array(labels)

    # Load and preprocess data
    X, y = load_and_preprocess_data(audio_dir, labels_df)

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    input_shape = X_train[0].shape
    input_layer = Input(shape=input_shape)

    # Convolutional layers
    x = Conv2D(32, (3, 3), activation='relu')(input_layer)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)

    # Fully connected layers
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    output_layer = Dense(len(classes), activation='softmax')(x)

    # Create model
    model = Model(inputs=input_layer, outputs=output_layer)

    from tensorflow.keras.losses import CategoricalCrossentropy

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss=CategoricalCrossentropy(),  # Log loss (categorical_crossentropy)
                  metrics=['accuracy'])

    history = model.fit(X_train, y_train,
                    epochs=16,
                    batch_size=32,
                    validation_data=(X_val, y_val))

    # Assuming 'history' contains training history (like from Keras)
    for epoch in range(len(history.history['accuracy'])):
       mlflow.log_metric("training_accuracy", history.history['accuracy'][epoch], step=epoch)
       mlflow.log_metric("validation_accuracy", history.history['val_accuracy'][epoch], step=epoch)
       mlflow.log_metric("training_loss", history.history['loss'][epoch], step=epoch)
       mlflow.log_metric("validation_loss", history.history['val_loss'][epoch], step=epoch)

    predictions = model.predict(X_val)


    # Log du modèle
    signature = infer_signature(X_train, predictions)
    print("Signature d'entrée : ", signature)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="base_model_13",
        signature=signature,
    )

    print("...Terminé !")
    print(f"Temps total d'entraînement : {time.time() - start_time:.2f}s")
    mlflow.log_param("Param name", "Value")



Accessing as ungjeanclaude

Initialized MLflow to track repo "ungjeanclaude/early-prediction-alzheimer"

Repository ungjeanclaude/early-prediction-alzheimer initialized!

2024/12/17 14:52:57 INFO mlflow.tracking.fluent: Experiment with name 'base_model_13' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/28dc563346ea4967aed62db54d0eae71', creation_time=1734447177779, experiment_id='9', last_update_time=1734447177779, lifecycle_stage='active', name='base_model_13', tags={}>
Début de l'entraînement du modèle...
Mounted at /content/drive
Epoch 1/16
42/42 ━━━━━━━━━━━━━━━━━━━━ 33s 714ms/step - accuracy: 0.4245 - loss: 52.0615 - val_accuracy: 0.5970 - val_loss: 0.9224
Epoch 2/16
42/42 ━━━━━━━━━━━━━━━━━━━━ 29s 689ms/step - accuracy: 0.5335 - loss: 0.9723 - val_accuracy: 0.6000 - val_loss: 0.9701
Epoch 3/16
42/42 ━━━━━━━━━━━━━━━━━━━━ 43s 730ms/step - accuracy: 0.5456 - loss: 0.9750 - val_accuracy: 0.6000 - val_loss: 0.8639
Epoch 4/16
42/42 ━━━━━━━━━━━━━━━━━━━━ 40s 730ms/step - accuracy: 0.5589 - loss: 0.9223 - val_accuracy: 0.5818 - val_loss: 0.9003
Epoch 5/16
42/42 ━━━━━━━━━━━━━━━━━━━━ 41s 730ms/step - accuracy: 0.5562 - loss: 0.8903 - val_accuracy: 0.5939 - val_loss: 0.8834
Epoch 6/16
42/42 ━━━━━━━━━━━━━━━━━━━━ 41s 724ms/step - accuracy: 0.57

2024/12/17 15:21:57 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2024-12-04; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'google-genai'}
Successfully registered model 'base_model_13'.
2024/12/17 15:22:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: base_model_13, version 1
Created version '1' of model 'base_model_13'.


...Terminé !
Temps total d'entraînement : 1743.30s
🏃 View run judicious-lamb-364 at: https://dagshub.com/ungjeanclaude/early-prediction-alzheimer.mlflow/#/experiments/9/runs/f6a906a4d584431a8d33ac0f4b015d31
🧪 View experiment at: https://dagshub.com/ungjeanclaude/early-prediction-alzheimer.mlflow/#/experiments/9
